In [1]:
import mlflow
import pandas as pd
import plotly.express as px
import dotenv
import os
from mlflow import MlflowClient
from typing import Optional
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from tqdm import tqdm

# Load environment variables from .env file
dotenv.load_dotenv()
mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")

# Set tracking URI (adjust if remote)
mlflow.set_tracking_uri(mlflow_uri)

# Choose experiment by name or ID
experiment_name = "STASC"
experiment = mlflow.get_experiment_by_name(experiment_name)

dr = mlflow.search_runs(experiment_ids=[experiment.experiment_id])


methods = ["cove", "baseline_cot", "baseline_no_cot", "stasc_FNF", "stasc_FNE", "stasc_FIF", "stasc_FIE", "stasc_ENF", "stasc_ENE", "stasc_EIF", "stasc_EIE"] 

In [97]:
client = MlflowClient(mlflow_uri)

df_final = pd.DataFrame()
for method in tqdm(methods[3:], desc="Experiments"):
    run_id = mlflow.search_runs(experiment_ids=[experiment.experiment_id], filter_string=f"run_name LIKE '{method}_%'", order_by=["start_time DESC"])["run_id"][0]
    child_runs = dr[dr["tags.mlflow.parentRunId"] == run_id]
    init_runs = child_runs[child_runs["tags.mlflow.runName"].str.contains("init_generation", na=False)]
    init_runs = init_runs[["run_id", "tags.mlflow.runName", "params.algo_name", "metrics.i_train_ACC", "metrics.i_test_ACC"]]
    init_runs = init_runs.rename(columns={"metrics.i_train_ACC": "metrics.c_train_ACC", "tags.mlflow.runName": "tags.mlflow.runName_x"})
    
    iteration_runs = child_runs[child_runs["tags.mlflow.runName"].str.contains("iteration", na=False)].sort_values("tags.mlflow.runName").reset_index(drop=True)
    fine_tune_runs = child_runs[child_runs["tags.mlflow.runName"].str.contains("fine-tune", na=False)].sort_values("tags.mlflow.runName").reset_index(drop=True)
    fine_tune_runs = fine_tune_runs[["tags.mlflow.runName", "metrics.i_test_ACC"]]
    iteration_runs = iteration_runs[["run_id", "tags.mlflow.runName", "params.algo_name", "metrics.c_train_ACC"]]
    iteration_runs["index"] = iteration_runs["tags.mlflow.runName"].str.extract(r'iteration_(\d+)', expand=False).astype(int)
    fine_tune_runs["index"] = fine_tune_runs["tags.mlflow.runName"].str.extract(r'fine-tune_(\d+)', expand=False).astype(int)
    
    df_sum = iteration_runs.merge(fine_tune_runs, on="index")
    df_sum = df_sum.drop(columns=["index"])

    df_final = pd.concat([df_final, df_sum, init_runs], ignore_index=True)


Experiments: 100%|██████████| 8/8 [00:00<00:00, 11.76it/s]


In [98]:

for suffix, metric, title, y_title in [
    ("_E", "metrics.c_train_ACC", "Training Accuracy over Iterations", "Train"),
    ("_F", "metrics.c_train_ACC", "Training Accuracy over Iterations", "Train"),
    ("_E", "metrics.i_test_ACC", "Test Accuracy over Iterations", "Test"),
    ("_F", "metrics.i_test_ACC", "Test Accuracy over Iterations", "Test")
]:
    df_plot = df_final.sort_values("tags.mlflow.runName_x")
    df_plot = df_plot[df_plot["params.algo_name"].str.contains(suffix, regex=True)]
    
    fig = px.line(df_plot, x="tags.mlflow.runName_x", y=metric,
                  color="params.algo_name", markers=True, title=title)
    
    fig.update_xaxes(title_text="")           # remove x-axis label
    fig.update_yaxes(title_text=y_title, range=[0.15, 0.22])
    fig.show()